In [2]:
# Install LangChain
!pip install langchain


# Import Libraries
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda


# -----------------------------
# Job Description
# -----------------------------
job_description = """
Python, Machine Learning, SQL, Deep Learning, NLP, TensorFlow, LangChain
Minimum 2 years experience
"""


# -----------------------------
# Resume Inputs
# -----------------------------
strong_resume = """
John Doe
5 years experience in Python, Machine Learning, SQL, Deep Learning,
NLP, TensorFlow, LangChain
Worked as Data Scientist
"""

average_resume = """
Jane Smith
2 years experience in Python, SQL, Machine Learning
Worked as Data Analyst
"""

weak_resume = """
Mike
Knows Excel, Communication, MS Office
Worked in Sales
"""


# -----------------------------
# Required Skills
# -----------------------------
required_skills = [
    "Python",
    "Machine Learning",
    "SQL",
    "Deep Learning",
    "NLP",
    "TensorFlow",
    "LangChain"
]


# -----------------------------
# Prompt Templates
# -----------------------------
extract_prompt = PromptTemplate.from_template(
    """
    Extract skills from the following resume:
    {resume}
    """
)

match_prompt = PromptTemplate.from_template(
    """
    Compare extracted skills with job description:
    Skills: {skills}
    JD: {jd}
    """
)

score_prompt = PromptTemplate.from_template(
    """
    Calculate score based on matching skills:
    {matching}
    """
)


# -----------------------------
# Skill Extraction Logic
# -----------------------------
def extract_logic(inputs):

    resume = inputs["resume"]

    found_skills = []

    for skill in required_skills:
        if skill.lower() in resume.lower():
            found_skills.append(skill)

    import re

    experience_match = re.search(r'(\d+)\s*years', resume)

    if experience_match:
        experience = int(experience_match.group(1))
    else:
        experience = 0

    return {
        "skills": found_skills,
        "experience": experience
    }


# -----------------------------
# Matching Logic
# -----------------------------
def match_logic(inputs):

    skills = inputs["skills"]

    matching = []
    missing = []

    for skill in required_skills:

        if skill in skills:
            matching.append(skill)

        else:
            missing.append(skill)

    return {
        "matching_skills": matching,
        "missing_skills": missing,
        "experience": inputs["experience"]
    }


# -----------------------------
# Scoring Logic
# -----------------------------
def score_logic(inputs):

    matching_count = len(inputs["matching_skills"])
    experience = inputs["experience"]

    score = (matching_count * 10) + (experience * 5)

    if score > 100:
        score = 100

    if score >= 80:
        explanation = "Excellent candidate with strong technical background."

    elif score >= 50:
        explanation = "Average candidate with moderate matching."

    else:
        explanation = "Weak candidate with insufficient required skills."

    return {
        "score": score,
        "explanation": explanation
    }


# -----------------------------
# LangChain Chains
# -----------------------------
extract_chain = RunnableLambda(extract_logic)

match_chain = RunnableLambda(match_logic)

score_chain = RunnableLambda(score_logic)


# -----------------------------
# Pipeline Function
# -----------------------------
def evaluate_resume(resume, candidate_type):

    print("\n" + "="*50)
    print(f"{candidate_type}")
    print("="*50)


    extracted = extract_chain.invoke({
        "resume": resume
    })

    print("\nExtracted Data:")
    print(extracted)


    matched = match_chain.invoke(extracted)

    print("\nMatching Result:")
    print(matched)


    scored = score_chain.invoke(matched)

    print("\nFinal Score:")
    print(scored)


# -----------------------------
# Run All Candidates
# -----------------------------
evaluate_resume(strong_resume, "Strong Candidate")

evaluate_resume(average_resume, "Average Candidate")

evaluate_resume(weak_resume, "Weak Candidate")


# -----------------------------
# Debug Incorrect Output
# -----------------------------
bad_resume = """
Worked on technical projects only
"""

evaluate_resume(bad_resume, "Incorrect Output Debug Test")


Strong Candidate

Extracted Data:
{'skills': ['Python', 'Machine Learning', 'SQL', 'Deep Learning', 'NLP', 'TensorFlow', 'LangChain'], 'experience': 5}

Matching Result:
{'matching_skills': ['Python', 'Machine Learning', 'SQL', 'Deep Learning', 'NLP', 'TensorFlow', 'LangChain'], 'missing_skills': [], 'experience': 5}

Final Score:
{'score': 95, 'explanation': 'Excellent candidate with strong technical background.'}

Average Candidate

Extracted Data:
{'skills': ['Python', 'Machine Learning', 'SQL'], 'experience': 2}

Matching Result:
{'matching_skills': ['Python', 'Machine Learning', 'SQL'], 'missing_skills': ['Deep Learning', 'NLP', 'TensorFlow', 'LangChain'], 'experience': 2}

Final Score:
{'score': 40, 'explanation': 'Weak candidate with insufficient required skills.'}

Weak Candidate

Extracted Data:
{'skills': [], 'experience': 0}

Matching Result:
{'matching_skills': [], 'missing_skills': ['Python', 'Machine Learning', 'SQL', 'Deep Learning', 'NLP', 'TensorFlow', 'LangChain'], '